Sebastián Toro Arango
#Optimization. Practical Tasks

Please, execute two rows of code below at first.

In [29]:
!pip install git+https://github.com/mehalyna/cooltest.git

  Cloning https://github.com/mehalyna/cooltest.git to /tmp/pip-req-build-irrzssiw
  Running command git clone --filter=blob:none --quiet https://github.com/mehalyna/cooltest.git /tmp/pip-req-build-irrzssiw
  Resolved https://github.com/mehalyna/cooltest.git to commit 630c96f2d3300782279879d5d13e6c1aaabf3c75
  Preparing metadata (setup.py) ... done


In [30]:
from cooltest.test_cool_3 import *

# Task 1. Resource Scheduling

In the resource scheduling task, we have a set of tasks to be performed, each with its own duration and resource requirements. Additionally, we have a set of available resources with limited capacity. The goal is to assign tasks to resources in such a way that all tasks are completed within their deadlines, and the resources are utilized efficiently without exceeding their capacities.

Your  task is to define `schedule_tasks()` function that takes the following inputs:

- `tasks`: A list of tuples representing tasks, where each tuple contains (`task_name`, `duration`, `resource_requirements`).
- `resources`: A dictionary representing available resources and their capacities, where the keys are resource names, and the values are their capacities.
- `deadline`: The maximum time (`deadline`) within which all tasks must be completed.

The function `schedule_tasks` returns a dictionary representing the optimal assignment of tasks to resources along with the completion time for each task.

>**Note**: implement a simple _Greedy Scheduling algorithm_ to optimize the resource scheduling task. In this algorithm, tasks are assigned to resources in a greedy manner based on their duration and resource requirements.

In [31]:
import copy

@test_schedule_task
def schedule_tasks(tasks, resources, deadline):
    """
    Optimize resource scheduling to complete tasks within the given deadline using Greedy Scheduling.
    """
    assigned_tasks = {}
    task_times = {}
    resource_times = {key: 0.0 for key in resources}
    available_resources = copy.deepcopy(resources)

    # Sort tasks in ascending order of their durations
    tasks.sort(key=lambda x: x[1])

    for task in tasks:
        task_name, duration, requirements = task
        assigned = False

        # Filter to only required resources, sorted by earliest finish time
        candidate_resources = sorted(
            [(r, resource_times[r]) for r in available_resources if requirements.get(r, 0) > 0],
            key=lambda x: x[1]
        )

        for resource_name, _ in candidate_resources:
            demand = requirements[resource_name]

            # If the resource doesn't have enough capacity for the task
            if available_resources[resource_name] < demand:
                continue

            # Check if task can be completed within deadline on this resource
            potential_completion_time = resource_times[resource_name] + duration

            if potential_completion_time <= deadline:
                # Assign task to the resource
                assigned_tasks.setdefault(resource_name, []).append(task_name)
                resource_times[resource_name] = potential_completion_time
                task_times[task_name] = potential_completion_time

                # Deduct capacity only from the assigned resource
                available_resources[resource_name] -= demand
                assigned = True
                break

    return {**assigned_tasks, **task_times}

# Example usage:
tasks_list = [
    ('TaskA', 4.5, {'Resource1': 2, 'Resource2': 1}),
    ('TaskB', 7.2, {'Resource2': 3}),
    ('TaskC', 5.0, {'Resource1': 1})
]
resources_dict = {'Resource1': 10, 'Resource2': 15}
deadline_time = 12.0

result = schedule_tasks(tasks_list, resources_dict, deadline_time)
print(result)

Schedule Task  Failed

{'Resource1': ['TaskA', 'TaskC'], 'Resource2': ['TaskB'], 'TaskA': 4.5, 'TaskC': 9.5, 'TaskB': 7.2}


# Task 2. Vehicle Routing Problem (VRP)

The **Vehicle Routing Problem (VRP)** is a classic optimization problem that involves a fleet of vehicles tasked with delivering goods or services to a set of customers from a central depot. Each customer has a demand for a certain quantity of goods, and the vehicles have limited capacities to carry these goods. The goal is to find the optimal set of routes for the vehicles such that all customers are visited exactly once, the total demand of each route does not exceed the vehicle capacity, and the overall travel time or distance is minimized.

Your next task is to define function `optimize_vrp()` that takes the following inputs:

- `depot`: The coordinates (x, y) of the depot where all vehicles start and end their routes.
- `customers`: A list of tuples representing customer locations and their demands, where each tuple contains (x, y, demand).
- `vehicle_capacity`: The maximum capacity of each vehicle.
- `num_vehicles`: The number of vehicles available in the fleet.

The function `optimize_vrp()` returns the optimized routes for the vehicles, along with the total travel distance.

Additionally you may define the function `calculate_distance()` and use it to calculate the distance between two locations.


> **Note:** The function will `optimize_vrp()` implement a brute-force approach to solve the Vehicle Routing Problem (VRP) and find the optimized routes for a fleet of vehicles to minimize travel distance. The function takes the depot location, customer locations and demands, vehicle capacity limit, and the number of available vehicles as input and returns the optimized routes for the vehicles along with the total travel distance. It uses brute force to generate all possible permutations of customer indices and evaluates the total travel distance for each permutation to find the best solution.

In [32]:
import itertools
import math

def calculate_distance(coord1, coord2):
    """
    Calculate the Euclidean distance between two points in 2D space.

    Args:
        coord1 (tuple): The coordinates (x, y) of the first point.
        coord2 (tuple): The coordinates (x, y) of the second point.

    Returns:
        float: The Euclidean distance between the two points.
    """
    x1, y1 = coord1
    x2, y2 = coord2
    return math.sqrt((x1 - x2) ** 2 + (y1 - y2) ** 2)

@test_optimize_vrp
def optimize_vrp(depot, customers, vehicle_capacity, num_vehicles):
    """
    Optimize the Vehicle Routing Problem to minimize total travel distance using Brute Force.

    Args:
        depot (tuple): The coordinates (x, y) of the depot, where the vehicles start and end their routes.
        customers (list of tuple): A list of tuples representing the coordinates (x, y) of each customer location.
        vehicle_capacity (int): The maximum capacity of each vehicle.
        num_vehicles (int): The number of vehicles available in the fleet.

    Returns:
        list: A list of routes, where each route represents the sequence of customer locations visited by a single vehicle.
    """
    num_customers = len(customers)
    all_permutations = list(itertools.permutations(range(num_customers)))

    best_distance = float('inf')
    best_routes = None

    for permutation in all_permutations:
        routes = [[] for _ in range(num_vehicles)]
        remaining_capacity = [vehicle_capacity] * num_vehicles
        current_locations = [depot] * num_vehicles
        total_distance = 0.0

        customer_locations_in_permutation = [customers[i] for i in permutation]

        for customer_location in customer_locations_in_permutation:
            min_distance = float('inf')
            best_veh = -1

            # Find the best vehicle based on its specific current location
            for vehicle in range(num_vehicles):
                if remaining_capacity[vehicle] >= 1:
                    dist = calculate_distance(current_locations[vehicle], customer_location)

                    if dist < min_distance:
                        min_distance = dist
                        best_veh = vehicle

            if best_veh != -1:
                routes[best_veh].append(customer_location)
                total_distance += min_distance
                remaining_capacity[best_veh] -= 1
                current_locations[best_veh] = customer_location

        # Add the return trip to the depot for each vehicle
        for vehicle in range(num_vehicles):
            if routes[vehicle]:
                total_distance += calculate_distance(current_locations[vehicle], depot)
                routes[vehicle].append(depot)

        if total_distance < best_distance:
            best_distance = total_distance
            best_routes = routes

    return best_routes

# Example usage:
depot_location = (0, 0)
customer_locations = [(1, 3), (3, 5), (4, 8), (9, 6), (7, 1)]
capacity_per_vehicle = 3
number_of_vehicles = 2

optimized_routes = optimize_vrp(depot_location, customer_locations, capacity_per_vehicle, number_of_vehicles)
print(optimized_routes)

VRP Task  Failed

[[(4, 8), (9, 6), (7, 1), (0, 0)], [(1, 3), (3, 5), (0, 0)]]


# Task 3. Inventory Management

**Inventory management** is the process of efficiently tracking and controlling the flow of goods or products in a business. The goal is to strike a balance between minimizing inventory costs and ensuring sufficient stock levels to meet customer demand. The inventory management problem involves determining the optimal inventory levels to minimize holding costs (costs associated with carrying inventory) while avoiding stockouts (running out of stock) and backorders (unfilled customer orders).

Your task is to define `optimize_inventory_management()` function that takes the following inputs:

- `demand`: A list representing the demand for each period (e.g., month, week) in the planning horizon.
- `holding_cost`: The cost of holding one unit of inventory for one period (e.g., month, week).
- `ordering_cost`: The cost of placing an order for a fixed quantity of inventory.
- `initial_inventory`: The initial inventory level at the beginning of the planning horizon.
- `reorder_point`: The inventory level at which a new order should be placed to avoid stockouts.

The function `optimize_inventory_management` should return a list representing the optimal inventory levels for each period in the planning horizon.

You have to use Linear Programming to find the optimal inventory levels for each period. The decision variables are the inventory levels and the order quantity for each period. The objective function aims to minimize the total cost, which includes both holding costs and ordering costs.

Constraints ensure that the inventory at the beginning of each period is sufficient to meet the demand and the reorder point constraint.

The PuLP library allows us to formulate the problem easily and efficiently. Once the Linear Programming problem is defined, we call model.solve() to find the optimal solution, and the optimal_inventory_levels list contains the optimal inventory levels for each period in the planning horizon.

_Linear Programming Model:_
Decision Variables:
- inventory[period]: The inventory level at the beginning of each period.
- order_quantity[period]: The order quantity placed at the beginning of each period.

Objective Function:
- Minimize the total cost, which includes holding costs and ordering costs for each period.

Constraints:
- `inventory[0] == initial_inventory`: Initial inventory level constraint.
- `inventory[period] >= demand[period] + order_quantity[period] - inventory[period - 1]`: Inventory balance constraint.
- `inventory[period] >= reorder_point`: Reorder point constraint.
- `inventory[period] >= 0 and order_quantity[period] >= 0`: Non-negativity constraints.

Note:
- The demand list should contain the demand for each period in the planning horizon.
- The `holding_cost` and `ordering_cost` are the costs per unit per period and per order, respectively.
- The `initial_inventory` is the initial inventory level at the beginning of the planning horizon.
- The `reorder_point` is the inventory level at which a new order should be placed.
- The function returns a list representing the optimal inventory levels for each period, including the initial period.

> The provided function will assume that the demand for each period is known in advance and does not consider uncertainty in demand forecasts. Additionally, it will assume that the inventory holding cost and ordering cost remain constant over the planning horizon. In real-world scenarios, demand may be uncertain, and costs may vary, so more sophisticated techniques like Stochastic Inventory Management or Dynamic Programming may be used for more complex inventory management problems.


In [33]:
!pip install pulp

In [34]:
import pulp

@test_optimize_oim
def optimize_inventory_management(demand, holding_cost, ordering_cost, initial_inventory, reorder_point):
    """
    Optimize inventory levels using Linear Programming.

    This function uses Linear Programming to determine the optimal inventory levels for each period in the planning
    horizon. The goal is to minimize the total cost of inventory while meeting the demand and inventory requirements.

    Args:
        demand (list): A list representing the demand for each period.
        holding_cost (float): The cost of holding one unit of inventory for one period.
        ordering_cost (float): The cost of placing an order for a fixed quantity of inventory.
        initial_inventory (int): The initial inventory level at the beginning of the planning horizon.
        reorder_point (int): The inventory level at which a new order should be placed to avoid stockouts.

    Returns:
        list: A list representing the optimal inventory levels for each period.
    """

    # Create a Linear Programming problem
    model = pulp.LpProblem("Inventory_Management", pulp.LpMinimize)

    # Decision variables
    periods = len(demand) + 1  # add 0 period
    demand_list = [0] + demand

    inventory = [pulp.LpVariable(f"inventory_{t}", lowBound=0) for t in range(periods)]
    order_quantity = [pulp.LpVariable(f"order_quantity_{t}", lowBound=0) for t in range(periods)]

    # Objective function: minimize total cost
    model += pulp.lpSum(holding_cost * inventory[t] + ordering_cost * order_quantity[t] for t in range(1, periods))

    # Constraints
    model += inventory[0] == initial_inventory

    for t in range(1, periods):
        model += inventory[t] == (inventory[t - 1] + order_quantity[t] - demand_list[t])
        model += order_quantity[t] >= (demand_list[t] - inventory[t - 1])
        model += order_quantity[t] >= reorder_point - inventory[t - 1]

    # Solve the Linear Programming problem
    model.solve()

    # Extract the optimal solution
    result = [inventory[t].varValue for t in range(periods)]

    # Information display
    info = [
        (f"Period {t}: "
         f"Starting inventory = {inventory[t-1].varValue}, "
         f"Order quantity = {order_quantity[t].varValue}, "
         f"Demand = {demand_list[t]}",
         f"Ending inventory = {inventory[t].varValue}, "
         f"Total cost = {holding_cost * inventory[t].varValue + ordering_cost * order_quantity[t].varValue}")
        for t in range(1, periods)
    ]

    print(*info, sep='\n')
    print(f"Model TOTAL cost = {pulp.value(model.objective)}")

    return result[1:]

# Example usage:
demand_forecast = [10, 20, 15, 25, 30]
holding_cost_per_period = 1.5
ordering_cost_per_order = 25.0
initial_inventory_level = 50
reorder_point_level = 50

optimal_inventory_levels = optimize_inventory_management(
    demand_forecast,
    holding_cost_per_period,
    ordering_cost_per_order,
    initial_inventory_level,
    reorder_point_level
)

print("Optimal Inventory Levels:", optimal_inventory_levels)

('Period 1: Starting inventory = 50.0, Order quantity = 0.0, Demand = 10', 'Ending inventory = 40.0, Total cost = 60.0')
('Period 2: Starting inventory = 40.0, Order quantity = 10.0, Demand = 20', 'Ending inventory = 30.0, Total cost = 295.0')
('Period 3: Starting inventory = 30.0, Order quantity = 20.0, Demand = 15', 'Ending inventory = 35.0, Total cost = 552.5')
('Period 4: Starting inventory = 35.0, Order quantity = 15.0, Demand = 25', 'Ending inventory = 25.0, Total cost = 412.5')
('Period 5: Starting inventory = 25.0, Order quantity = 25.0, Demand = 30', 'Ending inventory = 20.0, Total cost = 655.0')
Model TOTAL cost = 1975.0
OIM Task  Failed

('Period 1: Starting inventory = 50.0, Order quantity = 0.0, Demand = 10', 'Ending inventory = 40.0, Total cost = 60.0')
('Period 2: Starting inventory = 40.0, Order quantity = 10.0, Demand = 20', 'Ending inventory = 30.0, Total cost = 295.0')
('Period 3: Starting inventory = 30.0, Order quantity = 20.0, Demand = 15', 'Ending inventory = 35.